<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/Modell%20Finomhangol%C3%A1s%20LoRA%20Adapterek%20PEFT_M%C3%B3dszerek.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modell Finomhangolás: LoRA, Adapterek és más PEFT Módszerek

**Licenc: CC BY-NC-SA 4.0**

A **Parameter-Efficient Fine-Tuning (PEFT)** technikák lehetővé teszik nagy modellek hatékony adaptálását, a paraméterek töredékének módosításával.

## Miért kell PEFT?

| Full Fine-tuning | PEFT módszerek |
|------------------|----------------|
| 7B params = 28GB GPU memória | ~1% params, elfér 1 GPU-n |
| Minden súly frissül | Csak adapter/LoRA súlyok |
| Catastrophic forgetting | Stabil, eredeti tudás megmarad |
| Lassú tanítás | Gyors konvergencia |
| Sok tárolt modell (feladatonként) | Kis adapter fájlok |

## PEFT módszerek családja

```
PEFT
├── Additive (új paraméterek)
│   ├── Adapters
│   ├── Prefix Tuning
│   └── Prompt Tuning
├── Selective (létező paraméterek)
│   └── BitFit (csak bias)
└── Reparameterization
    ├── LoRA
    └── QLoRA (quantized)
```

## Tartalomjegyzék
1. LoRA (Low-Rank Adaptation)
2. Adapter Layers
3. Prefix Tuning
4. Módszerek összehasonlítása
5. Gyakorlati használat (Hugging Face PEFT)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

## 1. LoRA (Low-Rank Adaptation)

A **LoRA** (Hu et al., 2021) az egyik legnépszerűbb PEFT módszer. Az ötlet: a súlymátrix változása **alacsony rangú**.

### Matematikai háttér

Az eredeti súly $W_0$ fagyasztva marad, és egy alacsony rangú update-et tanulunk:

$$W_{new} = W_{frozen} + \Delta W = W_0 + BA$$

Ahol:
- $W_0 \in \mathbb{R}^{d \times k}$ - eredeti (fagyasztott) súlyok
- $B \in \mathbb{R}^{d \times r}$ - down projection
- $A \in \mathbb{R}^{r \times k}$ - up projection
- $r \ll \min(d, k)$ - rank (tipikusan 4, 8, 16)

### Paraméter megtakarítás

Eredeti: $d \times k$ paraméter
LoRA: $d \times r + r \times k = r(d + k)$ paraméter

Ha $d = k = 768$ és $r = 8$:
- Eredeti: 589,824 params
- LoRA: 12,288 params (2.1%)

### Hol alkalmazzuk?

Tipikusan az **attention súlyokra**: $W_q$, $W_k$, $W_v$, $W_o$

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base_layer, rank=4, alpha=1.0):
        super().__init__()
        self.base = base_layer
        self.base.weight.requires_grad = False  # Freeze

        d_out, d_in = base_layer.weight.shape
        self.lora_A = nn.Parameter(torch.randn(d_in, rank) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(rank, d_out))
        self.scale = alpha / rank

    def forward(self, x):
        base_out = self.base(x)
        lora_out = (x @ self.lora_A @ self.lora_B) * self.scale
        return base_out + lora_out

# Példa
base = nn.Linear(768, 768)
lora = LoRALinear(base, rank=8)

total_params = sum(p.numel() for p in base.parameters())
trainable = sum(p.numel() for p in lora.parameters() if p.requires_grad)
print(f"Eredeti: {total_params:,} params")
print(f"LoRA trainable: {trainable:,} params ({100*trainable/total_params:.2f}%)")

## 2. Adapter Layers

Az **Adapter** (Houlsby et al., 2019) megközelítés új, kis bottleneck rétegeket szúr be a Transformer blokkokba.

### Architektúra

```
Original layer output
        ↓
    Layer Norm
        ↓
  Down projection (d → bottleneck)
        ↓
    Nonlinearity (GELU)
        ↓
  Up projection (bottleneck → d)
        ↓
    Residual (+)
        ↓
     Output
```

### Elhelyezés

Tipikusan minden Transformer blokkban **két adapter**:
1. Self-attention után
2. FFN után

### LoRA vs Adapter

| | LoRA | Adapter |
|--|------|---------|
| Új rétegek | Nem | Igen |
| Inference overhead | Nincs (merge-elhető) | Van |
| Paraméterek | Kevesebb | Több |

In [ ]:
class Adapter(nn.Module):
    def __init__(self, d_model, bottleneck=64):
        super().__init__()
        self.down = nn.Linear(d_model, bottleneck)
        self.up = nn.Linear(bottleneck, d_model)
        self.act = nn.GELU()

    def forward(self, x):
        return x + self.up(self.act(self.down(x)))

class TransformerWithAdapter(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, 4, batch_first=True)
        self.adapter1 = Adapter(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, 1024), nn.ReLU(), nn.Linear(1024, d_model))
        self.adapter2 = Adapter(d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):
        h, _ = self.attn(x, x, x)
        x = self.ln1(x + self.adapter1(h))
        h = self.ffn(x)
        x = self.ln2(x + self.adapter2(h))
        return x

model = TransformerWithAdapter()
print(f"Adapter params: {sum(p.numel() for n, p in model.named_parameters() if 'adapter' in n):,}")

## 3. Prefix Tuning

A **Prefix Tuning** (Li & Liang, 2021) virtuális tokeneket tanul, amelyek az input elé kerülnek.

### Működés

```
Eredeti: [User input tokens...]
Prefix:  [Learned prefix...] [User input tokens...]
```

A prefix vektorok **feladatspecifikusak** és befolyásolják az attention-t.

### Előnyök

- Nagyon kevés paraméter (~0.1%)
- Nem módosítja a modell súlyait
- Könnyű váltani feladatok között

### Hátrányok

- Csökkenti az effektív context length-et
- Néha instabil tanulás

In [ ]:
class PrefixTuning(nn.Module):
    def __init__(self, prefix_len=10, d_model=256):
        super().__init__()
        self.prefix = nn.Parameter(torch.randn(prefix_len, d_model) * 0.01)

    def forward(self, x):
        # Prefix hozzáfűzése az inputhoz
        B = x.shape[0]
        prefix = self.prefix.unsqueeze(0).expand(B, -1, -1)
        return torch.cat([prefix, x], dim=1)

prefix = PrefixTuning(prefix_len=20, d_model=256)
x = torch.randn(2, 50, 256)
print(f"Input: {x.shape} → With prefix: {prefix(x).shape}")
print(f"Prefix params: {prefix.prefix.numel():,}")

## 4. Módszerek összehasonlítása

### Mikor melyiket használd?

| Módszer | Paraméterek | Use case |
|---------|-------------|----------|
| **Full FT** | 100% | Sok adat, erős GPU |
| **LoRA** | 0.1-1% | Általános célú, ajánlott |
| **QLoRA** | 0.1% + quant | Limitált GPU memória |
| **Adapter** | 2-5% | Ha kell inference-nél is kontroll |
| **Prefix** | 0.1% | Nagyon kevés paraméter kell |

In [ ]:
methods = ['Full FT', 'LoRA r=8', 'LoRA r=16', 'Adapter', 'Prefix']
params_pct = [100, 0.5, 1.0, 2.0, 0.1]

plt.figure(figsize=(8, 4))
plt.bar(methods, params_pct, color=['red', 'green', 'green', 'blue', 'orange'])
plt.ylabel('Trainable params %')
plt.title('PEFT módszerek összehasonlítása')
plt.yscale('log')
for i, v in enumerate(params_pct):
    plt.text(i, v*1.2, f'{v}%', ha='center')
plt.show()

## 5. Gyakorlati használat (Hugging Face PEFT)

A `peft` library egyszerűvé teszi a PEFT módszerek használatát.

### LoRA konfiguráció

```python
LoraConfig(
    r=8,              # Rank (4, 8, 16, 32)
    lora_alpha=32,    # Scaling factor
    target_modules=["q_proj", "v_proj"],  # Melyik rétegekre
    lora_dropout=0.1, # Regularizáció
    bias="none"       # Bias tanulás (none/all/lora_only)
)
```

### Training tippek

1. **Learning rate**: Magasabb mint full FT (1e-4 vs 2e-5)
2. **Rank választás**: Kezdj r=8-cal, növeld ha kell
3. **Target modules**: Attention + MLP is segíthet
4. **Alpha**: Gyakran `alpha = 2 * r`

In [ ]:
print("""
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b")

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# trainable params: 4,194,304 || all params: 6,742,609,920 || trainable%: 0.062
""")

## Összefoglalás

### PEFT módszerek áttekintése

| Módszer | Trainable % | Hol módosít | Inference overhead |
|---------|-------------|-------------|-------------------|
| **LoRA** | ~0.1-1% | Attention weights | Nincs (merge) |
| **QLoRA** | ~0.1% | LoRA + 4-bit quant | Nincs |
| **Adapter** | ~2-5% | Új rétegek | Van |
| **Prefix** | ~0.1% | Input prefix | Minimális |
| **Prompt Tuning** | ~0.01% | Soft prompt | Minimális |

### Főbb tanulságok

1. **LoRA** a legelterjedtebb és legjobb kompromisszum
2. **QLoRA** ha limitált a GPU memória
3. A PEFT módszerek **közel full FT teljesítményt** érnek el
4. **Adapter fájlok** kicsik, könnyen megoszthatók

### Ajánlott workflow

```
1. Válaszd ki a base modellt (LLaMA, Mistral, stb.)
2. Konfiguráld a LoRA-t (r=8, target=q,v proj)
3. Készítsd elő a training adatot
4. Fine-tune 1-3 epoch
5. Merge a LoRA weights-et (opcionális)
6. Deploy
```

### Következő lépések

A következő notebookban az **LLM Agents** témát nézzük, ahol az LLM-ek eszközöket használnak és komplex feladatokat oldanak meg!